# IS403.R11 – Nhóm 6: S05/S06 và C04
Đọc đủ 5 file M5 gốc, ghi số dòng/cột và đo Seasonal Naive trên 10 chuỗi.
C04 chỉ dùng lịch sử Train `d_1–d_1885`, chu kỳ 7 ngày, dự báo 28 ngày.
Notebook dùng CPU. Kết quả được lưu riêng theo môi trường; việc chốt mẫu C05 vẫn cần số đo của các mô hình khác.


In [ ]:
import os, sys, subprocess, platform, json, zipfile, gc
from pathlib import Path
from datetime import datetime, timezone, timedelta

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    runtime, base = 'kaggle', Path('/kaggle/working')
elif 'COLAB_RELEASE_TAG' in os.environ:
    runtime, base = 'colab', Path('/content')
else:
    raise RuntimeError('Chạy notebook này trên Colab hoặc Kaggle; dùng lệnh README trên máy cá nhân.')
ROOT = base / 'IS403.R11-M5-Competition'
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/haiphamt/IS403.R11-M5-Competition.git', str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'PyYAML'], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Runtime:', runtime)


## Dữ liệu
**Colab:** chạy ô dưới và chọn ZIP tải bằng **Download all files → Download as zip** trên trang M5.
**Kaggle:** thêm cuộc thi **M5 Forecasting – Accuracy** vào Input; ô dưới tự bỏ qua bước tải lên.
Dữ liệu gốc chỉ nằm trong phiên notebook và không được đưa lên Git.


In [ ]:
if runtime == 'colab':
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Chọn đúng một ZIP chứa đủ 5 file M5 gốc.')
    names = ['calendar.csv', 'sell_prices.csv', 'sales_train_validation.csv', 'sales_train_evaluation.csv', 'sample_submission.csv']
    with zipfile.ZipFile(next(iter(uploaded))) as archive:
        if not set(names).issubset(archive.namelist()):
            raise ValueError('ZIP thiếu file M5 hoặc file CSV không nằm ngay ở thư mục gốc.')
        archive.extractall(ROOT / 'data' / 'raw', members=names)
    del uploaded
    gc.collect()
else:
    print('Kaggle: đọc bộ M5 đã gắn trong Input.')


In [ ]:
import pandas as pd
import numpy as np
import yaml
from m5.data import load_raw_data

raw = load_raw_data()
shapes = pd.DataFrame([{'table': name, 'rows': len(table), 'columns': len(table.columns)} for name, table in raw.items()])
print(shapes.to_string(index=False))
out = ROOT / 'results' / 'tables'
out.mkdir(parents=True, exist_ok=True)
shapes.to_csv(out / f's05_data_shapes_{runtime}.csv', index=False)
environment = {
    'task': 'S05/S06', 'runtime': runtime, 'all_rows_read': True,
    'recorded_at': datetime.now(timezone(timedelta(hours=7))).isoformat(),
    'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'pyyaml': yaml.__version__,
    'os': platform.system(), 'logical_cpu_count': os.cpu_count(),
    'commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'source': 'https://www.kaggle.com/competitions/m5-forecasting-accuracy/data'
}
(out / f's05_environment_{runtime}.json').write_text(json.dumps(environment, indent=2) + '\n', encoding='utf-8')
print(json.dumps(environment, indent=2))
subprocess.run(['free', '-h'], check=True)
subprocess.run(['nproc'], check=True)
del raw
gc.collect()
subprocess.run([sys.executable, 'scripts/benchmark_seasonal_naive.py'], check=True)

RESULT_ZIP = base / f'is403_m5_checks_{runtime}.zip'
with zipfile.ZipFile(RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(out.glob(f'*_{runtime}.*')):
        archive.write(path, arcname=path.name)
print('Results:', RESULT_ZIP)


In [ ]:
if runtime == 'colab':
    files.download(str(RESULT_ZIP))
else:
    print('Tải ZIP kết quả từ Output của notebook:', RESULT_ZIP.name)
